# Fase B — Limpieza y Normalización de Datos
### Proyecto 2 y 3 — Análisis, Limpieza y Modelado · Dataset GEIH (DANE), periodo 202606
Introducción a la Ciencia de Datos — Universidad del Valle

Este notebook toma el dataset explorado en la Fase A (una fila por persona encuestada) y deja un DataFrame **limpio y listo para modelar**, guardado en `entrega/datos_limpios.parquet`.

**Contenido**
0. Configuración y parámetros
1. Carga (misma lógica de la Fase A, conservando las llaves para diagnosticar duplicados)
2. Diagnóstico: duplicados, nulos (estructurales vs. reales), texto, códigos especiales y atípicos
3. Limpieza determinista (reglas que no aprenden nada de los datos)
4. Partición train/test y tratamiento de nulos reales
5. Atípicos y transformación logarítmica
6. Codificación de categóricas (ordinal y one-hot)
7. Escalado (ajustado solo con train)
8. Guardado y verificación
9. Tabla de decisiones (base del documento breve)

> **Nombres:** las variables y funciones usan el prefijo `geih_2026` / `GEIH_2026` para ser consistentes con la Fase A.
>
> **Regla anti data leakage:** todo lo que *aprende* de los datos (medianas, modas, escaladores) se calcula **solo con las filas de train**. Por eso la partición se hace en la sección 4, antes de imputar y escalar, y la columna `particion` queda guardada en el parquet para que la Fase C use exactamente el mismo split.
>
> **Antes de ejecutar:** tener los CSV de la GEIH en la raíz del proyecto (local) o en `MyDrive/GEIH` (Colab) y usar *Restart & Run All*.

## 0. Configuración y parámetros

In [ ]:
%pip install -q scikit-learn pyarrow matplotlib seaborn

In [ ]:
import os, re, shutil, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler

# GEIH_2026: estilo común, igual al de la Fase A, para que todas las figuras del informe se vean consistentes
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titleweight"] = "bold"
GEIH_2026_COP = mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")  # formato moneda para ejes

# GEIH_2026: una sola semilla para todo lo aleatorio; la Fase C debe usar la misma
GEIH_2026_SEMILLA = 42

In [ ]:
# GEIH_2026: misma carga que la Fase A, pero funciona en Colab y en local
try:
    from google.colab import drive
    drive.mount("/content/drive")
    GEIH_2026_RUTA = "/content/drive/MyDrive/GEIH"
    GEIH_2026_RAIZ = "."
    GEIH_2026_EN_COLAB = True
except ImportError:
    # GEIH_2026: en local, si el notebook corre dentro de entrega/ (lo habitual en VS Code), los CSV están un nivel arriba
    GEIH_2026_CWD = os.path.abspath(os.getcwd())
    GEIH_2026_RAIZ = (os.path.dirname(GEIH_2026_CWD)
                      if os.path.basename(GEIH_2026_CWD).lower().startswith("entrega")   # "entrega", "entrega 1", "entrega 2"...
                      else GEIH_2026_CWD)
    GEIH_2026_RUTA = GEIH_2026_RAIZ   # CSV en la raíz del proyecto, como en el Proyecto 1
    GEIH_2026_EN_COLAB = False

# GEIH_2026: parámetros de limpieza, todos juntos para poder justificarlos y cambiarlos en un solo lugar
GEIH_2026_CODIGOS_NS_OTROS = [98, 99]                       # "no sabe / no informa" en columnas de otros ingresos (verificar en el diccionario DANE)
GEIH_2026_CODIGOS_NS = {"semanas_buscando_trabajo": [998, 999]}   # códigos que no son semanas reales (verificar en el diccionario)
GEIH_2026_MAX_HORAS = 112       # 16 h diarias x 7 días: más que eso se trata como error de captura
GEIH_2026_TEST_SIZE = 0.20      # partición 80/20
GEIH_2026_SALIDA = os.path.join(GEIH_2026_RAIZ, "entrega")   # carpeta donde se guardan los parquet

## 1. Carga del dataset

Se reutiliza la lógica de la Fase A (6 CSV de la GEIH unidos en una fila por persona). Dos diferencias, ambas por la limpieza:

1. Las **llaves** (`DIRECTORIO`, `SECUENCIA_P`, `ORDEN`, `HOGAR`) se conservan hasta diagnosticar duplicados; después ya no se puede saber si dos filas iguales son la misma persona.
2. En las columnas de **otros ingresos**, los códigos 98/99 se convierten en nulo **antes de sumarlas**; si no, quedan sumados como si fueran pesos (en la Fase A el mínimo de `ingreso_otras_fuentes` era 98).

In [ ]:
GEIH_2026_CLAVES = ["DIRECTORIO", "SECUENCIA_P", "ORDEN"]
GEIH_2026_CLAVES_HOGAR = ["DIRECTORIO", "SECUENCIA_P", "HOGAR"]

def geih_2026_normalizar(texto):
    """GEIH_2026: minúsculas, sin tildes, y con '_' y espacios unificados, para comparar nombres de archivo y de categorías."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[\s_\-]+", " ", t).strip()

def geih_2026_buscar(nombre):
    """GEIH_2026: devuelve la ruta del CSV pedido. Prioriza el nombre exacto (ignorando tildes, guiones y mayúsculas),
    luego los que empiezan igual y por último los que lo contienen; así 'Ocupados' no se confunde con 'No ocupados'."""
    clave = geih_2026_normalizar(nombre)
    exactos, empiezan, contienen = [], [], []
    for raiz, carpetas, archivos in os.walk(GEIH_2026_RUTA):
        carpetas[:] = [c for c in carpetas if not c.startswith(".") and c != "__pycache__"]   # no entrar a .venv, .git, etc.
        for a in archivos:
            if not a.lower().endswith(".csv"):
                continue
            stem = geih_2026_normalizar(os.path.splitext(a)[0])
            ruta = os.path.join(raiz, a)
            if stem == clave:
                exactos.append(ruta)
            elif stem.startswith(clave):
                empiezan.append(ruta)
            elif clave in stem:
                contienen.append(ruta)
    for grupo in (exactos, empiezan, contienen):
        if grupo:
            return sorted(grupo, key=len)[0]
    raise FileNotFoundError(f"No encontré ningún CSV llamado '{nombre}' dentro de {GEIH_2026_RUTA}.")

def geih_2026_leer(nombre, columnas):
    """GEIH_2026: lee un CSV de la GEIH (sep ';') buscándolo por nombre y cargando solo las columnas pedidas."""
    ruta = geih_2026_buscar(nombre)
    try:
        return pd.read_csv(ruta, sep=";", encoding="utf-8", usecols=lambda c: c in columnas, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(ruta, sep=";", encoding="latin-1", usecols=lambda c: c in columnas, low_memory=False)

In [ ]:
CL, CH = GEIH_2026_CLAVES, GEIH_2026_CLAVES_HOGAR
geih_2026_num = lambda s: pd.to_numeric(s, errors="coerce")

# GEIH_2026: lectura de los 6 CSV necesarios (mismas columnas que la Fase A)
geih_2026_ft = geih_2026_leer("Fuerza de trabajo", CL + ["HOGAR", "FT", "FFT"])
geih_2026_car = geih_2026_leer("Características generales", CL + ["DPTO", "P6040", "P3271", "P6070", "P3042"])
geih_2026_hog = geih_2026_leer("Datos del hogar", CH + ["P4010", "P5000", "P6008", "P4030S1", "P4030S2", "P4030S3", "P4030S4", "P4030S5"])
geih_2026_ocu = geih_2026_leer("Ocupados", CL + ["INGLABO", "P6800"])
geih_2026_noc = geih_2026_leer("No ocupados", CL + ["P7250"])
_cols_otros = ["P7510S1A1", "P7510S2A1", "P7510S3A1", "P7510S5A1", "P7510S6A1", "P7510S7A1",
               "P750S1A1", "P750S2A1", "P750S3A1", "P1661S3A1", "P1661S4A1", "P1661S5A1", "P1661S6A1"]
geih_2026_oti = geih_2026_leer("Otros ingresos", CL + _cols_otros)

# GEIH_2026: estado ocupacional (Ocupado si está en Ocupados; si no, FT=1 desempleado, FFT=1 inactivo)
geih_2026_base = geih_2026_ft.drop_duplicates(subset=CL).copy()
geih_2026_base["es_ocupado"] = geih_2026_base.set_index(CL).index.isin(
    geih_2026_ocu.drop_duplicates(subset=CL).set_index(CL).index)
geih_2026_base["estado_ocupacional"] = np.select(
    [geih_2026_base["es_ocupado"], geih_2026_num(geih_2026_base["FT"]) == 1,
     geih_2026_num(geih_2026_base["FFT"]) == 1],
    ["Ocupado", "Desempleado", "Inactivo"], default="Sin clasificar")

# GEIH_2026: acceso a servicios del hogar (Completo / Parcial / Ninguno), igual que en la Fase A
_serv = geih_2026_hog[[c for c in geih_2026_hog.columns if c.startswith("P4030S")]].apply(geih_2026_num)
geih_2026_hog["acceso_servicios"] = np.select([(_serv == 1).all(axis=1), (_serv == 1).any(axis=1)],
                                                   ["Completo", "Parcial"], default="Ninguno")
geih_2026_hog = geih_2026_hog.drop(columns=_serv.columns).rename(
    columns={"P4010": "tipo_vivienda", "P5000": "cuartos", "P6008": "personas_hogar"})

# GEIH_2026: otros ingresos. Primero los códigos 98/99 pasan a nulo y recién después se suma
_oti = geih_2026_oti.copy()
_cols_ok = [c for c in _cols_otros if c in _oti.columns]
_oti[_cols_ok] = _oti[_cols_ok].apply(geih_2026_num)
_suma_antes = _oti[_cols_ok].sum(axis=1, min_count=1)
geih_2026_n_codigos_otros = int(_oti[_cols_ok].isin(GEIH_2026_CODIGOS_NS_OTROS).sum().sum())
_oti[_cols_ok] = _oti[_cols_ok].replace(GEIH_2026_CODIGOS_NS_OTROS, np.nan)
_oti["ingreso_otras_fuentes"] = _oti[_cols_ok].sum(axis=1, min_count=1)
print(f"Valores 98/99 convertidos a nulo en otros ingresos: {geih_2026_n_codigos_otros:,}")
print(f"Mínimo de ingreso_otras_fuentes: antes {_suma_antes.min():,.0f} -> después {_oti['ingreso_otras_fuentes'].min():,.0f}")

# GEIH_2026: unión final, una fila por persona; se conservan las llaves para el diagnóstico de duplicados
geih_2026_df_crudo = (
    geih_2026_base[CL + ["HOGAR", "estado_ocupacional"]]
    .merge(geih_2026_car.drop_duplicates(subset=CL).rename(
        columns={"DPTO": "departamento", "P6040": "edad", "P3271": "sexo", "P6070": "estado_civil", "P3042": "nivel_educativo"}),
        on=CL, how="inner")
    .merge(geih_2026_hog.drop_duplicates(subset=CH), on=CH, how="left")
    .merge(geih_2026_ocu.drop_duplicates(subset=CL).rename(columns={"INGLABO": "ingreso_laboral", "P6800": "horas_trabajadas"}), on=CL, how="left")
    .merge(geih_2026_noc.drop_duplicates(subset=CL).rename(columns={"P7250": "semanas_buscando_trabajo"}), on=CL, how="left")
    .merge(_oti[CL + ["ingreso_otras_fuentes"]].drop_duplicates(subset=CL), on=CL, how="left")
    .reset_index(drop=True))
geih_2026_df_crudo.insert(0, "id_hecho", range(1, len(geih_2026_df_crudo) + 1))
print(f"Filas: {geih_2026_df_crudo.shape[0]:,} | Columnas: {geih_2026_df_crudo.shape[1]}")

In [ ]:
GEIH_2026_DEPARTAMENTOS = {
    "05": "Antioquia", "08": "Atlántico", "11": "Bogotá D.C.", "13": "Bolívar", "15": "Boyacá",
    "17": "Caldas", "18": "Caquetá", "19": "Cauca", "20": "Cesar", "23": "Córdoba",
    "25": "Cundinamarca", "27": "Chocó", "41": "Huila", "44": "La Guajira", "47": "Magdalena",
    "50": "Meta", "52": "Nariño", "54": "Norte de Santander", "63": "Quindío", "66": "Risaralda",
    "68": "Santander", "70": "Sucre", "73": "Tolima", "76": "Valle del Cauca", "81": "Arauca",
    "85": "Casanare", "86": "Putumayo", "88": "San Andrés y Providencia", "91": "Amazonas",
    "94": "Guainía", "95": "Guaviare", "97": "Vaupés", "99": "Vichada",
}

# GEIH_2026: agrupación del nivel educativo (códigos P3042 de la GEIH), la misma de la Fase A.
# IMPORTANTE: verificar contra el diccionario del DANE, porque el Ordinal Encoding de la sección 6 depende de este orden.
GEIH_2026_NIVEL_EDU = {
    1: "Sin educación", 2: "Sin educación", 3: "Primaria", 4: "Secundaria",
    5: "Media", 6: "Media", 7: "Media", 8: "Técnico/Tecnológico", 9: "Técnico/Tecnológico",
    10: "Universitario", 11: "Posgrado", 12: "Posgrado", 13: "Posgrado",
}
GEIH_2026_ORDEN_EDU = ["Sin educación", "Primaria", "Secundaria", "Media",
                            "Técnico/Tecnológico", "Universitario", "Posgrado"]
GEIH_2026_ORDEN_SERVICIOS = ["Ninguno", "Parcial", "Completo"]

def geih_2026_normalizar_departamento(valor):
    """GEIH_2026: devuelve el nombre del departamento.
    En la bodega algunos departamentos quedaron como código sin cero a la izquierda ('5', '8')
    y otros ya como nombre; aquí se unifican ambos casos."""
    if pd.isna(valor):
        return np.nan
    texto = str(valor).strip()
    try:
        codigo = str(int(float(texto))).zfill(2)
    except ValueError:
        return texto.title()
    return GEIH_2026_DEPARTAMENTOS.get(codigo, texto)

# GEIH_2026: ajuste de tipos y etiquetas (igual que la Fase A; todavía no se elimina ni se imputa nada)
geih_2026_num_cols = ["ingreso_laboral", "ingreso_otras_fuentes", "horas_trabajadas",
                           "semanas_buscando_trabajo", "edad", "cuartos", "personas_hogar"]
for col in geih_2026_num_cols:
    geih_2026_df_crudo[col] = pd.to_numeric(geih_2026_df_crudo[col], errors="coerce")

geih_2026_df_crudo["sexo"] = pd.to_numeric(geih_2026_df_crudo["sexo"], errors="coerce").map({1: "Hombre", 2: "Mujer"})
geih_2026_df_crudo["nivel_educativo_grupo"] = (
    pd.to_numeric(geih_2026_df_crudo["nivel_educativo"], errors="coerce").map(GEIH_2026_NIVEL_EDU))
geih_2026_df_crudo["departamento"] = geih_2026_df_crudo["departamento"].map(geih_2026_normalizar_departamento)
geih_2026_df_crudo.drop(columns=CL + ["HOGAR"]).dtypes.to_frame("dtype")

## 2. Diagnóstico

Se revisan cuatro cosas antes de tocar nada: **duplicados**, **nulos**, **inconsistencias de texto** y **atípicos**. La idea es decidir con evidencia y no a ojo.

### 2.1 Duplicados
Se miran en las fuentes originales (por llave) y en el DataFrame unido. Esto se hace **antes** de quitar las llaves.

In [ ]:
# GEIH_2026: filas repetidas por llave en cada CSV original (la Fase A ya las quita con drop_duplicates al unir)
geih_2026_dup_fuentes = pd.Series({
    "Fuerza de trabajo": int(geih_2026_ft.duplicated(subset=CL).sum()),
    "Características generales": int(geih_2026_car.duplicated(subset=CL).sum()),
    "Ocupados": int(geih_2026_ocu.duplicated(subset=CL).sum()),
    "No ocupados": int(geih_2026_noc.duplicated(subset=CL).sum()),
    "Otros ingresos": int(geih_2026_oti.duplicated(subset=CL).sum()),
    "Datos del hogar (por hogar)": int(geih_2026_hog.duplicated(subset=CH).sum()),
}, name="filas_duplicadas_por_llave")
display(geih_2026_dup_fuentes.to_frame())

# GEIH_2026: duplicados en el DataFrame unido
geih_2026_dup_llave = int(geih_2026_df_crudo.duplicated(subset=CL).sum())
geih_2026_dup_filas = int(geih_2026_df_crudo.drop(columns=CL + ["HOGAR", "id_hecho"]).duplicated().sum())
print(f"Personas repetidas por llave (DIRECTORIO, SECUENCIA_P, ORDEN): {geih_2026_dup_llave}")
print(f"Filas idénticas en todas las variables (sin contar llaves): {geih_2026_dup_filas}")

**Decisión sobre duplicados:** una persona es única por su llave (`DIRECTORIO` + `SECUENCIA_P` + `ORDEN`), y el DataFrame unido tiene 0 repetidas porque la carga ya elimina las repetidas por llave en cada fuente. Las filas idénticas en *todas* las variables (si aparecen) **no se eliminan**: son personas distintas del mismo hogar o de hogares parecidos que coinciden por casualidad, no registros copiados. Eliminarlas borraría información real.

In [ ]:
# GEIH_2026: a partir de aquí se trabaja sin llaves (ya no hacen falta) y con una bandera de ocupado
geih_2026_df = geih_2026_df_crudo.drop(columns=CL + ["HOGAR"]).copy()
geih_2026_df["es_ocupado"] = geih_2026_df["estado_ocupacional"] == "Ocupado"
print(f"Filas: {len(geih_2026_df):,} | Columnas: {geih_2026_df.shape[1]}")
geih_2026_df["estado_ocupacional"].value_counts(normalize=True).mul(100).round(1).to_frame("%")

### 2.2 Nulos: estructurales vs. reales
No todo nulo es un error. Un nulo es **estructural** cuando la pregunta no aplica a esa persona (por ejemplo, el ingreso laboral de un inactivo) y **real** cuando debería existir y falta. Cada tipo se trata distinto.

In [ ]:
def geih_2026_clasificar_nulos(df, reglas):
    """GEIH_2026: separa los nulos de cada columna en estructurales y reales.
    `reglas` asocia cada columna con una máscara booleana de las filas donde el dato SÍ debería existir.
    Un nulo en una fila donde la máscara es False es estructural; si la máscara es True, es real."""
    filas = []
    for col, aplica in reglas.items():
        nulo = df[col].isna()
        filas.append({"columna": col, "nulos_total": int(nulo.sum()),
                      "estructurales": int((nulo & ~aplica).sum()),
                      "reales": int((nulo & aplica).sum())})
    salida = pd.DataFrame(filas).set_index("columna")
    for c in ["nulos_total", "estructurales", "reales"]:
        salida["pct_" + c.replace("nulos_", "")] = (salida[c] / len(df) * 100).round(2)
    return salida

geih_2026_todos = pd.Series(True, index=geih_2026_df.index)
geih_2026_reglas = {
    "semanas_buscando_trabajo": geih_2026_df["estado_ocupacional"] == "Desempleado",   # solo aplica a quien busca empleo
    "ingreso_otras_fuentes": pd.Series(False, index=geih_2026_df.index),               # vacío = "no declaró otros ingresos", no es un dato perdido
    "ingreso_laboral": geih_2026_df["es_ocupado"],                                     # solo ocupados
    "horas_trabajadas": geih_2026_df["es_ocupado"],                                    # solo ocupados
    "edad": geih_2026_todos, "sexo": geih_2026_todos, "departamento": geih_2026_todos,
    "estado_civil": geih_2026_todos, "nivel_educativo": geih_2026_todos,
    "tipo_vivienda": geih_2026_todos, "cuartos": geih_2026_todos,
    "personas_hogar": geih_2026_todos, "acceso_servicios": geih_2026_todos,
}
geih_2026_nulos = geih_2026_clasificar_nulos(geih_2026_df, geih_2026_reglas)
display(geih_2026_nulos.sort_values("nulos_total", ascending=False))

# GEIH_2026: figura de nulos estructurales (gris) y reales (rojo), como % del total de personas
geih_2026_nul = geih_2026_nulos[geih_2026_nulos["nulos_total"] > 0].sort_values("nulos_total")
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(geih_2026_nul.index, geih_2026_nul["pct_estructurales"], color="#94a3b8",
        label="Estructurales (la pregunta no aplica)")
ax.barh(geih_2026_nul.index, geih_2026_nul["pct_reales"], left=geih_2026_nul["pct_estructurales"],
        color="#ef4444", label="Reales (debería existir y falta)")
ax.set_title("Nulos por columna: estructurales vs. reales")
ax.set_xlabel("Porcentaje de personas (%)")
ax.set_ylabel("Variable")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("geih_2026_nulos_estructurales_reales.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** casi todos los vacíos son estructurales. `ingreso_laboral` y `horas_trabajadas` solo existen para ocupados, y `semanas_buscando_trabajo` solo para desempleados, así que **no deben imputarse con una media**: inventarían ingresos para inactivos y, además, delatarían el estado ocupacional. Los nulos *reales* (rojo) son pocos y se concentran en ocupados sin ingreso declarado. En `ingreso_otras_fuentes` el vacío significa "no declaró otros ingresos" (no es un dato perdido), y se trata aparte en la sección 3.

### 2.3 Inconsistencias de texto

In [ ]:
def geih_2026_diag_texto(df, columnas):
    """GEIH_2026: detecta variantes de escritura de una misma categoría (mayúsculas, tildes, espacios).
    Si `n_unicos_normalizados` es menor que `n_unicos`, hay categorías duplicadas por escritura."""
    filas = []
    for col in columnas:
        s = df[col].dropna().astype(str)
        limpio = s.str.strip().map(geih_2026_normalizar)
        filas.append({"columna": col, "n_unicos": int(s.nunique()), "n_unicos_normalizados": int(limpio.nunique()),
                      "con_espacios_extremos": int((s != s.str.strip()).sum()),
                      "ejemplos": ", ".join(sorted(s.unique())[:5])})
    return pd.DataFrame(filas).set_index("columna")

display(geih_2026_diag_texto(geih_2026_df, ["sexo", "departamento", "estado_ocupacional", "acceso_servicios", "nivel_educativo_grupo"]))

# GEIH_2026: departamentos que no están en la lista oficial (códigos sin mapear o escrituras raras)
geih_2026_extra = sorted(set(geih_2026_df["departamento"].dropna()) - set(GEIH_2026_DEPARTAMENTOS.values()))
print("Departamentos fuera de la lista oficial:", geih_2026_extra if geih_2026_extra else "ninguno")

**Decisión sobre texto:** el único problema real de texto era `departamento` (códigos con y sin cero a la izquierda, y nombres ya escritos), y ya se unificó con `geih_2026_normalizar_departamento`. Si la celda anterior lista departamentos fuera de la lista oficial, hay que revisar su código antes de seguir.

### 2.4 Códigos especiales que parecen números
En la GEIH, valores como 98/99 o 998/999 suelen ser "no sabe / no informa", no cantidades. Se cuentan para decidir. **Ojo:** 98 o 99 pueden ser valores reales en horas o semanas, por eso se confirma contra el diccionario del DANE.

In [ ]:
# GEIH_2026: filas con códigos tipo "no sabe / no informa" que quedaron como números reales
geih_2026_codigos = [98, 99, 998, 999]
geih_2026_cols_codigos = ["ingreso_otras_fuentes", "semanas_buscando_trabajo", "ingreso_laboral", "horas_trabajadas"]
geih_2026_df_codigos = pd.DataFrame(
    {c: geih_2026_df[c].isin(geih_2026_codigos).sum() for c in geih_2026_cols_codigos},
    index=["filas_con_codigo"]).T
display(geih_2026_df_codigos)
print("Máximo de semanas_buscando_trabajo:", geih_2026_df["semanas_buscando_trabajo"].max())

### 2.5 Atípicos: IQR y Z-score

In [ ]:
def geih_2026_resumen_atipicos(df, columnas, solo_positivos=()):
    """GEIH_2026: cuenta atípicos por la regla del rango intercuartílico (IQR).
    Un valor es atípico si queda por encima de Q3 + 1.5*IQR o por debajo de Q1 - 1.5*IQR.
    Las columnas en `solo_positivos` se evalúan solo con valores > 0 (ingresos)."""
    filas = []
    for col in columnas:
        serie = df[col].dropna()
        if col in solo_positivos:
            serie = serie[serie > 0]
        q1, q3 = serie.quantile(.25), serie.quantile(.75)
        iqr = q3 - q1
        li, ls = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((serie < li) | (serie > ls)).sum())
        filas.append({"variable": col, "n": len(serie), "q1": q1, "q3": q3, "limite_superior": ls,
                      "n_atipicos": n_out, "pct_atipicos": round(n_out / len(serie) * 100, 1)})
    return pd.DataFrame(filas).set_index("variable")

geih_2026_atipicos = geih_2026_resumen_atipicos(
    geih_2026_df, geih_2026_num_cols, solo_positivos=("ingreso_laboral", "ingreso_otras_fuentes"))
display(geih_2026_atipicos)

# GEIH_2026: Z-score como segundo criterio. El ingreso se mide en log10 porque es muy asimétrico
geih_2026_ing_pos = geih_2026_df.loc[geih_2026_df["ingreso_laboral"] > 0, "ingreso_laboral"]
geih_2026_z_ing = np.log10(geih_2026_ing_pos)
geih_2026_z_ing = (geih_2026_z_ing - geih_2026_z_ing.mean()) / geih_2026_z_ing.std()
geih_2026_horas = geih_2026_df["horas_trabajadas"].dropna()
geih_2026_z_hor = (geih_2026_horas - geih_2026_horas.mean()) / geih_2026_horas.std()
print(f"Ingreso laboral con |Z| > 3 (en log10): {(geih_2026_z_ing.abs() > 3).sum():,} de {len(geih_2026_z_ing):,}")
print(f"Horas trabajadas con |Z| > 3: {(geih_2026_z_hor.abs() > 3).sum():,} de {len(geih_2026_z_hor):,}")
print("\nLos 10 ingresos laborales más altos:")
print(geih_2026_ing_pos.sort_values(ascending=False).head(10).map(lambda v: f"${v:,.0f}").to_string())

**Insight:** el IQR marca como atípico a más de 1 de cada 7 trabajadores en horas (jornadas largas, sobre todo informales) y a cerca de 8 de cada 100 en ingreso laboral (personas bien pagadas). Compara esos conteos con los del Z-score (en log10 para el ingreso): si el Z-score marca muchos menos casos, parte de lo que el IQR llama "atípico" es solo la cola larga normal de un ingreso asimétrico. Revisa también la lista de los ingresos más altos: si hay valores repetidos o redondos muy extremos, conviene verificarlos. La decisión sobre cada variable se toma en las secciones 3 y 5.

## 3. Limpieza determinista

Son reglas que **no aprenden nada de los datos** (no calculan medias ni medianas), por lo que se pueden aplicar a todo el DataFrame sin riesgo de leakage.

In [ ]:
# GEIH_2026: 3.1 códigos que no son semanas reales pasan a nulo
for _col, _cods in GEIH_2026_CODIGOS_NS.items():
    _mask = geih_2026_df[_col].isin(_cods)
    print(f"{_col}: {int(_mask.sum()):,} valores {_cods} convertidos a nulo")
    geih_2026_df.loc[_mask, _col] = np.nan

# GEIH_2026: 3.2 horas imposibles (más de GEIH_2026_MAX_HORAS a la semana) pasan a nulo; luego se imputan en la sección 4
geih_2026_horas_antes = geih_2026_df["horas_trabajadas"].copy()
geih_2026_mask_horas = geih_2026_df["horas_trabajadas"] > GEIH_2026_MAX_HORAS
geih_2026_n_horas_imposibles = int(geih_2026_mask_horas.sum())
geih_2026_df.loc[geih_2026_mask_horas, "horas_trabajadas"] = np.nan
print(f"horas_trabajadas: {geih_2026_n_horas_imposibles:,} valores > {GEIH_2026_MAX_HORAS} h convertidos a nulo")

# GEIH_2026: 3.3 ingreso de otras fuentes: vacío significa "no declaró otros ingresos"
#     -> se crea una bandera 0/1 y el monto vacío se rellena con 0 (regla estructural, no estadística)
geih_2026_df["recibe_otros_ingresos"] = geih_2026_df["ingreso_otras_fuentes"].notna().astype(int)
geih_2026_df["ingreso_otras_fuentes"] = geih_2026_df["ingreso_otras_fuentes"].fillna(0)
print(f"recibe_otros_ingresos = 1 en {geih_2026_df['recibe_otros_ingresos'].mean() * 100:.1f}% de las personas")

# GEIH_2026: 3.4 estado civil y tipo de vivienda son códigos de categoría, no cantidades -> se tratan como texto
for _col in ["estado_civil", "tipo_vivienda"]:
    geih_2026_df[_col] = pd.to_numeric(geih_2026_df[_col], errors="coerce").round().astype("Int64").astype("string")

In [ ]:
# GEIH_2026: figura de horas trabajadas antes y después de tratar las imposibles (solo ocupados)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
sns.histplot(geih_2026_horas_antes.dropna(), bins=40, color="#f97316", ax=axes[0])
axes[0].axvline(GEIH_2026_MAX_HORAS, color="black", linestyle="--", label=f"Límite: {GEIH_2026_MAX_HORAS} h")
axes[0].set_title("Horas trabajadas — antes")
axes[0].set_xlabel("Horas trabajadas por semana")
axes[0].set_ylabel("Número de personas")
axes[0].legend()
sns.histplot(geih_2026_df["horas_trabajadas"].dropna(), bins=40, color="#2563eb", ax=axes[1])
axes[1].set_title("Horas trabajadas — después")
axes[1].set_xlabel("Horas trabajadas por semana")
axes[1].set_ylabel("Número de personas")
fig.suptitle("Tratamiento de horas imposibles (GEIH, periodo 202606)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("geih_2026_horas_antes_despues.png", dpi=150, bbox_inches="tight")
plt.show()

**Decisión sobre horas:** se usa el IQR solo para *marcar*. Las jornadas de 61 a 112 horas se **conservan** porque son frecuentes en el trabajo informal y eliminarlas borraría justo a quienes más importan para el negocio. Solo se tratan como error las de más de 112 horas semanales (16 h diarias los 7 días), que además la Fase A ya había señalado como poco creíbles (hasta 130 h). Esos valores se vuelven nulo y se imputan en la sección 4.

## 4. Partición train/test y tratamiento de nulos reales

Aquí se hace la partición **antes** de calcular cualquier mediana o moda. Así los valores de relleno salen solo de train y test sigue siendo "datos que el modelo nunca vio". La partición queda guardada en la columna `particion` para que la Fase C use exactamente el mismo split.

In [ ]:
# GEIH_2026: partición 80/20 estratificada por estado ocupacional (mantiene las proporciones en ambos lados)
geih_2026_idx_train, geih_2026_idx_test = train_test_split(
    geih_2026_df.index, test_size=GEIH_2026_TEST_SIZE,
    random_state=GEIH_2026_SEMILLA, stratify=geih_2026_df["estado_ocupacional"])
geih_2026_df["particion"] = "train"
geih_2026_df.loc[geih_2026_idx_test, "particion"] = "test"
geih_2026_es_train = geih_2026_df["particion"] == "train"
print(geih_2026_df["particion"].value_counts().to_frame("personas"))
print(geih_2026_df.groupby("particion")["estado_ocupacional"].value_counts(normalize=True).mul(100).round(1).unstack())

In [ ]:
# GEIH_2026: imputación de nulos reales. Todos los valores de relleno se calculan SOLO con train.
geih_2026_log_imputacion = []
_ocup = geih_2026_df["es_ocupado"]

# 4.1 Estrategia MEDIANA (numérica con atípicos): horas trabajadas de ocupados con dato faltante o imposible.
#     Se imputa solo entre ocupados: para el resto el nulo es estructural y se deja como está.
geih_2026_med_horas = geih_2026_df.loc[geih_2026_es_train & _ocup, "horas_trabajadas"].median()
_mask = _ocup & geih_2026_df["horas_trabajadas"].isna()
geih_2026_df.loc[_mask, "horas_trabajadas"] = geih_2026_med_horas
geih_2026_log_imputacion.append({"columna": "horas_trabajadas", "estrategia": "mediana de train (solo ocupados)",
                                      "valor": float(geih_2026_med_horas), "n_imputados": int(_mask.sum())})

# 4.2 Estrategia MEDIANA también para las demás numéricas sin nulos esperados (protege el pipeline si cambian los datos)
for _col in ["edad", "cuartos", "personas_hogar"]:
    _med = geih_2026_df.loc[geih_2026_es_train, _col].median()
    _mask = geih_2026_df[_col].isna()
    geih_2026_df.loc[_mask, _col] = _med
    geih_2026_log_imputacion.append({"columna": _col, "estrategia": "mediana de train",
                                          "valor": float(_med), "n_imputados": int(_mask.sum())})

# 4.3 Estrategia MODA (categóricas): se rellena con la categoría más frecuente de train
for _col in ["sexo", "departamento", "estado_civil", "tipo_vivienda", "acceso_servicios", "nivel_educativo_grupo"]:
    _moda = geih_2026_df.loc[geih_2026_es_train, _col].mode(dropna=True)
    _moda = _moda.iloc[0] if len(_moda) else None
    _mask = geih_2026_df[_col].isna()
    if _moda is not None:
        geih_2026_df.loc[_mask, _col] = _moda
    geih_2026_log_imputacion.append({"columna": _col, "estrategia": "moda de train",
                                          "valor": str(_moda), "n_imputados": int(_mask.sum())})

geih_2026_df_log_imp = pd.DataFrame(geih_2026_log_imputacion).set_index("columna")
display(geih_2026_df_log_imp)

# GEIH_2026: el objetivo (ingreso_laboral) NO se imputa. Los ocupados sin ingreso o con ingreso 0 se excluyen de la muestra de modelado
geih_2026_n_sin_ingreso = int((_ocup & geih_2026_df["ingreso_laboral"].isna()).sum())
geih_2026_n_ingreso_cero = int((_ocup & (geih_2026_df["ingreso_laboral"] == 0)).sum())
geih_2026_df["muestra_modelado"] = _ocup & geih_2026_df["ingreso_laboral"].notna() & (geih_2026_df["ingreso_laboral"] > 0)
print(f"Ocupados sin ingreso declarado: {geih_2026_n_sin_ingreso:,}")
print(f"Ocupados con ingreso 0: {geih_2026_n_ingreso_cero:,}")
print(f"Muestra de modelado (ocupados con ingreso > 0): {int(geih_2026_df['muestra_modelado'].sum()):,} personas")

**Decisiones de nulos (resumen):**
- **Mediana** para `horas_trabajadas`: tiene muchos atípicos, y la media se contaminaría con las jornadas más largas. La mediana es robusta y mantiene el valor típico (jornada completa).
- **Moda** para las categóricas: no existe "promedio" de una categoría; la más frecuente es la apuesta menos distorsionante. Si la tabla `geih_2026_df_log_imp` muestra 0 imputados en una columna, la estrategia queda como salvaguarda: esa columna no tenía nulos reales.
- **Regla estructural** (sección 3) para `ingreso_otras_fuentes` y para los nulos de `semanas_buscando_trabajo` / `horas_trabajadas` fuera de su población: no se imputan, porque no son datos perdidos.
- **No se imputa el objetivo** (`ingreso_laboral`): rellenarlo contaminaría la evaluación del modelo. Los ocupados sin ingreso declarado quedan fuera de `muestra_modelado`, igual que los que tienen ingreso 0, que suelen ser ocupados sin remuneración (otro proceso, no comparable con un salario).
- **KNN no se usa:** con tan pocos nulos reales no aporta frente a la mediana y es mucho más costoso en 53 mil filas.

## 5. Atípicos y transformación logarítmica

Los atípicos de ingreso **se conservan** (son personas reales y bien pagadas, como concluyó la Fase A) y se controla su efecto con una transformación logarítmica. Se dejan además dos columnas bandera para poder analizarlos aparte si hace falta.

In [ ]:
# GEIH_2026: banderas de atípico por IQR (solo informativas; no modifican ningún valor)
def geih_2026_bandera_iqr(serie, solo_positivos=False):
    """GEIH_2026: devuelve 1 si el valor supera Q3 + 1.5*IQR (o queda bajo Q1 - 1.5*IQR), 0 en caso contrario.
    Los límites se calculan con la serie completa y solo sirven para marcar, no para cambiar datos (no hay leakage)."""
    base = serie[serie > 0] if solo_positivos else serie.dropna()
    q1, q3 = base.quantile(.25), base.quantile(.75)
    iqr = q3 - q1
    return ((serie > q3 + 1.5 * iqr) | (serie < q1 - 1.5 * iqr)).fillna(False).astype(int)

geih_2026_df["ingreso_laboral_atipico"] = geih_2026_bandera_iqr(geih_2026_df["ingreso_laboral"], solo_positivos=True)
geih_2026_df["horas_atipico"] = geih_2026_bandera_iqr(geih_2026_df["horas_trabajadas"].where(_ocup))
print(geih_2026_df[["ingreso_laboral_atipico", "horas_atipico"]].mean().mul(100).round(1).to_frame("% marcado"))

# GEIH_2026: transformación logarítmica con log1p (acepta ceros y deja los nulos como nulos)
geih_2026_df["log_ingreso_laboral"] = np.log1p(geih_2026_df["ingreso_laboral"])
geih_2026_df["log_ingreso_otras_fuentes"] = np.log1p(geih_2026_df["ingreso_otras_fuentes"])

geih_2026_skew = pd.DataFrame({
    "asimetría_original": [geih_2026_df.loc[geih_2026_df["muestra_modelado"], "ingreso_laboral"].skew(),
                           geih_2026_df.loc[geih_2026_df["ingreso_otras_fuentes"] > 0, "ingreso_otras_fuentes"].skew()],
    "asimetría_log1p": [geih_2026_df.loc[geih_2026_df["muestra_modelado"], "log_ingreso_laboral"].skew(),
                        geih_2026_df.loc[geih_2026_df["ingreso_otras_fuentes"] > 0, "log_ingreso_otras_fuentes"].skew()],
}, index=["ingreso_laboral", "ingreso_otras_fuentes"]).round(2)
display(geih_2026_skew)

# GEIH_2026: asimetría de TODAS las numéricas (|asimetría| > 1 = muy sesgada). Justifica qué variables se transforman y cuáles no
geih_2026_skew_todas = pd.Series({
    "ingreso_laboral (ocupados con ingreso > 0)": geih_2026_df.loc[geih_2026_df["muestra_modelado"], "ingreso_laboral"].skew(),
    "ingreso_otras_fuentes (solo > 0)": geih_2026_df.loc[geih_2026_df["ingreso_otras_fuentes"] > 0, "ingreso_otras_fuentes"].skew(),
    "horas_trabajadas (ocupados)": geih_2026_df.loc[_ocup, "horas_trabajadas"].skew(),
    "semanas_buscando_trabajo (con dato)": geih_2026_df["semanas_buscando_trabajo"].skew(),
    "edad": geih_2026_df["edad"].skew(),
    "cuartos": geih_2026_df["cuartos"].skew(),
    "personas_hogar": geih_2026_df["personas_hogar"].skew(),
}, name="asimetria").round(2).to_frame()
geih_2026_skew_todas["muy_sesgada (|a| > 1)"] = geih_2026_skew_todas["asimetria"].abs() > 1
display(geih_2026_skew_todas)

In [ ]:
# GEIH_2026: figura de ingreso laboral original vs. log1p (muestra de modelado)
_ing = geih_2026_df.loc[geih_2026_df["muestra_modelado"]]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(_ing["ingreso_laboral"], bins=60, color="#f97316", ax=axes[0])
axes[0].xaxis.set_major_formatter(GEIH_2026_COP)
axes[0].set_title("Ingreso laboral — escala original")
axes[0].set_xlabel("Ingreso laboral mensual (COP)")
axes[0].set_ylabel("Número de personas")
sns.histplot(_ing["log_ingreso_laboral"], bins=60, color="#2563eb", ax=axes[1])
axes[1].set_title("Ingreso laboral — log1p")
axes[1].set_xlabel("log(1 + ingreso laboral)")
axes[1].set_ylabel("Número de personas")
fig.suptitle("Efecto de la transformación logarítmica (ocupados con ingreso, GEIH 202606)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("geih_2026_ingreso_original_vs_log.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** en escala original casi toda la gente se amontona a la izquierda y unos pocos ingresos muy altos estiran la cola; con `log1p` la distribución queda mucho más simétrica y la asimetría baja de forma notable (tabla anterior). La tabla de asimetría de todas las numéricas justifica qué se transforma: el logaritmo se aplica a los dos ingresos, que son las variables muy sesgadas que entran como objetivo o predictora. Si `semanas_buscando_trabajo` aparece como muy sesgada, se deja sin transformar porque no entra como predictora en los modelos de ingreso. Se elige `log1p` en lugar de Box-Cox o Yeo-Johnson porque acepta ceros, no necesita ajustar parámetros con los datos (cero riesgo de leakage) y se interpreta directo: una diferencia en log es, aproximadamente, un cambio porcentual del ingreso. Cuando el modelo prediga en log, la Fase C debe devolver los resultados a pesos con `np.expm1` para reportar el error en una unidad entendible.

## 6. Codificación de categóricas

- **Ordinal Encoding** para variables con orden natural: `acceso_servicios` (Ninguno < Parcial < Completo) y `nivel_educativo` (de *Sin educación* a *Posgrado*). Un número ordenado conserva esa jerarquía, y un one-hot la perdería.
- **One-Hot Encoding** (`pd.get_dummies(..., drop_first=True)`) para variables nominales sin orden: `sexo`, `departamento`, `estado_civil` y `tipo_vivienda`. El `drop_first=True` elimina una categoría de referencia para evitar la multicolinealidad perfecta que rompe a los modelos lineales.

In [ ]:
# GEIH_2026: 6.1 Ordinal Encoding con orden explícito (0 = nivel más bajo)
geih_2026_df["acceso_servicios_ord"] = geih_2026_df["acceso_servicios"].map(
    {v: i for i, v in enumerate(GEIH_2026_ORDEN_SERVICIOS)})
geih_2026_df["nivel_educativo_ord"] = geih_2026_df["nivel_educativo_grupo"].map(
    {v: i for i, v in enumerate(GEIH_2026_ORDEN_EDU)})
assert geih_2026_df[["acceso_servicios_ord", "nivel_educativo_ord"]].notna().all().all(), "Hay categorías sin orden definido"
geih_2026_df[["acceso_servicios_ord", "nivel_educativo_ord"]] = geih_2026_df[["acceso_servicios_ord", "nivel_educativo_ord"]].astype(int)

def geih_2026_nombre_columna(nombre):
    """GEIH_2026: nombre de columna en minúsculas, sin tildes ni espacios (apto para parquet y para los modelos)."""
    t = unicodedata.normalize("NFKD", str(nombre)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", t).strip("_")

# GEIH_2026: 6.2 One-Hot Encoding con drop_first=True. Las columnas originales de texto se reemplazan por sus dummies
geih_2026_cols_ohe = ["sexo", "departamento", "estado_civil", "tipo_vivienda"]
geih_2026_df_enc = pd.get_dummies(
    geih_2026_df.drop(columns=["acceso_servicios", "nivel_educativo", "nivel_educativo_grupo"]),
    columns=geih_2026_cols_ohe, drop_first=True, dtype=int)
geih_2026_df_enc.columns = [geih_2026_nombre_columna(c) for c in geih_2026_df_enc.columns]
print(f"Columnas después de codificar: {geih_2026_df_enc.shape[1]}")
print("Dummies creadas:", [c for c in geih_2026_df_enc.columns if c.startswith(("sexo_", "estado_civil_", "tipo_vivienda_"))])
print("Departamentos (dummies):", sum(c.startswith("departamento_") for c in geih_2026_df_enc.columns))

**Nota:** `nivel_educativo_ord` usa la agrupación de la Fase A; si al verificar el diccionario del DANE cambia algún grupo, solo hay que corregir `GEIH_2026_NIVEL_EDU` y volver a ejecutar. Se descartó el código original `nivel_educativo` (1–13) para no tener dos versiones de la misma variable en el modelo.

## 7. Escalado (ajustado solo con train)

Cada escalador se elige según la forma de la variable:

| Escalador | Variables | Por qué |
|---|---|---|
| `RobustScaler` | `horas_trabajadas`, `cuartos`, `personas_hogar` | Usan mediana e IQR, así que los atípicos conservados no distorsionan la escala |
| `StandardScaler` (Z-score) | `edad`, `nivel_educativo_ord`, `acceso_servicios_ord` | Distribuciones razonablemente simétricas, sin atípicos fuertes |
| `MinMaxScaler` | `log_ingreso_otras_fuentes` | Alrededor del 75 % son ceros, así que Q1 y Q3 valen 0 (o casi) y el IQR queda en ~0; un RobustScaler casi no escalaría nada. Min-Max lleva todo a [0, 1] |

El **objetivo** (`log_ingreso_laboral`), las dummies y las banderas 0/1 **no se escalan**.

In [ ]:
GEIH_2026_COLS_ROBUSTO = ["horas_trabajadas", "cuartos", "personas_hogar"]
GEIH_2026_COLS_ESTANDAR = ["edad", "nivel_educativo_ord", "acceso_servicios_ord"]
GEIH_2026_COLS_MINMAX = ["log_ingreso_otras_fuentes"]
geih_2026_cols_escalar = GEIH_2026_COLS_ROBUSTO + GEIH_2026_COLS_ESTANDAR + GEIH_2026_COLS_MINMAX

# GEIH_2026: la versión escalada es una copia; los escaladores se ajustan SOLO con las filas de train
geih_2026_df_escalado = geih_2026_df_enc.copy()
geih_2026_df_escalado[geih_2026_cols_escalar] = geih_2026_df_escalado[geih_2026_cols_escalar].astype(float)
geih_2026_es_train_enc = geih_2026_df_escalado["particion"] == "train"

for _escalador, _cols in [(RobustScaler(), GEIH_2026_COLS_ROBUSTO),
                          (StandardScaler(), GEIH_2026_COLS_ESTANDAR),
                          (MinMaxScaler(), GEIH_2026_COLS_MINMAX)]:
    _escalador.fit(geih_2026_df_escalado.loc[geih_2026_es_train_enc, _cols])
    geih_2026_df_escalado[_cols] = _escalador.transform(geih_2026_df_escalado[_cols])

# GEIH_2026: verificación. En train el Z-score da media ~0 y desviación ~1; en test se parece pero no es exacto, y eso es lo correcto
geih_2026_verif = pd.concat({
    "train": geih_2026_df_escalado.loc[geih_2026_es_train_enc, geih_2026_cols_escalar].agg(["mean", "std", "min", "max"]).T,
    "test": geih_2026_df_escalado.loc[~geih_2026_es_train_enc, geih_2026_cols_escalar].agg(["mean", "std", "min", "max"]).T,
}, axis=1).round(3)
display(geih_2026_verif)

In [ ]:
# GEIH_2026: figura antes/después del escalado para las variables con escalas más distintas
_vars = ["horas_trabajadas", "edad", "log_ingreso_otras_fuentes"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for j, _v in enumerate(_vars):
    sns.boxplot(x=geih_2026_df_enc.loc[geih_2026_df_enc["muestra_modelado"], _v], color="#f97316", ax=axes[0, j])
    axes[0, j].set_title(f"{_v} — antes")
    axes[0, j].set_xlabel("Valor original")
    sns.boxplot(x=geih_2026_df_escalado.loc[geih_2026_df_escalado["muestra_modelado"], _v], color="#2563eb", ax=axes[1, j])
    axes[1, j].set_title(f"{_v} — después")
    axes[1, j].set_xlabel("Valor escalado")
fig.suptitle("Escalado de variables numéricas (muestra de modelado)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("geih_2026_escalado_antes_despues.png", dpi=150, bbox_inches="tight")
plt.show()

**Insight:** antes del escalado las variables viven en escalas muy distintas (horas en decenas, edad en decenas pero con otra dispersión, ingreso en logaritmo). Después, todas quedan comparables, lo que importa para modelos basados en distancia (KNN, K-Means, SVM) y para los de gradiente. En test los estadísticos no son exactamente 0 y 1, y eso es **la señal de que no hubo leakage**: el escalador no vio esas filas al ajustarse.

## 8. Guardado y verificación

Se guardan **dos** archivos con las mismas filas y columnas:

- `entrega/datos_limpios.parquet`: limpio y codificado, **sin escalar**. Es el que la Fase C debe usar con un `Pipeline` que ajuste el escalador dentro de train (es lo que pide el enunciado: no normalizar antes del split).
- `entrega/datos_limpios_escalado.parquet`: la misma tabla con el escalado de la sección 7, ajustado solo con train, para ver el resultado final de la normalización.

Las columnas quedan en cuatro grupos, en este orden:
- **Control** (no son predictoras): `id_hecho`, `particion`, `estado_ocupacional`, `es_ocupado`, `muestra_modelado`.
- **Objetivo y derivadas del objetivo** (nunca como predictoras): `ingreso_laboral`, `log_ingreso_laboral`, `ingreso_laboral_atipico`. El objetivo propuesto es `log_ingreso_laboral` (sobre `muestra_modelado`).
- **Referencia** (no entran como predictoras): `semanas_buscando_trabajo` (solo existe para desempleados) e `ingreso_otras_fuentes` (se usa su versión logarítmica).
- **Predictoras**: todas las demás; la celda de verificación imprime la lista y comprueba que no tengan nulos en la muestra de modelado.

In [ ]:
# GEIH_2026: orden de columnas: control primero, luego objetivo y variables originales, y al final las predictoras codificadas
GEIH_2026_COLS_CONTROL = ["id_hecho", "particion", "estado_ocupacional", "es_ocupado", "muestra_modelado"]
GEIH_2026_COLS_OBJETIVO = ["ingreso_laboral", "log_ingreso_laboral", "ingreso_laboral_atipico"]   # el objetivo y lo derivado de él: NUNCA como predictoras
GEIH_2026_COLS_REFERENCIA = ["semanas_buscando_trabajo", "ingreso_otras_fuentes"]   # se dejan como referencia; no entran como predictoras
geih_2026_predictoras = [c for c in geih_2026_df_enc.columns
                         if c not in GEIH_2026_COLS_CONTROL + GEIH_2026_COLS_OBJETIVO + GEIH_2026_COLS_REFERENCIA]
geih_2026_orden = GEIH_2026_COLS_CONTROL + GEIH_2026_COLS_OBJETIVO + GEIH_2026_COLS_REFERENCIA + geih_2026_predictoras
geih_2026_df_limpio = geih_2026_df_enc[geih_2026_orden].copy()
geih_2026_df_limpio_esc = geih_2026_df_escalado[geih_2026_orden].copy()

os.makedirs(GEIH_2026_SALIDA, exist_ok=True)
geih_2026_ruta_limpio = os.path.join(GEIH_2026_SALIDA, "datos_limpios.parquet")
geih_2026_ruta_esc = os.path.join(GEIH_2026_SALIDA, "datos_limpios_escalado.parquet")
geih_2026_df_limpio.to_parquet(geih_2026_ruta_limpio, index=False)
geih_2026_df_limpio_esc.to_parquet(geih_2026_ruta_esc, index=False)

# GEIH_2026: verificación leyendo de nuevo el archivo guardado
geih_2026_rel = pd.read_parquet(geih_2026_ruta_limpio)
assert geih_2026_rel.shape == geih_2026_df_limpio.shape, "El parquet guardado no coincide con el DataFrame"
print(f"{geih_2026_ruta_limpio}: {geih_2026_rel.shape[0]:,} filas x {geih_2026_rel.shape[1]} columnas")
print(f"{geih_2026_ruta_esc}: guardado")
print("\nRUTA COMPLETA del archivo que lee la Fase C:", os.path.abspath(geih_2026_ruta_limpio))
print(f"Columnas predictoras ({len(geih_2026_predictoras)}):", geih_2026_predictoras)

# GEIH_2026: en la muestra de modelado no debe quedar ningún nulo en las predictoras
_pred = geih_2026_predictoras
_nulos_pred = geih_2026_rel.loc[geih_2026_rel["muestra_modelado"], _pred].isna().sum().sum()
assert _nulos_pred == 0, f"Quedan {_nulos_pred} nulos en las predictoras de la muestra de modelado"
print("Nulos en predictoras (muestra de modelado): 0")
print("Nulos que quedan (estructurales, o el objetivo y semanas, que por decisión no se imputan):")
display(geih_2026_rel.isna().sum()[lambda s: s > 0].to_frame("nulos"))

# GEIH_2026: en Colab, además se copian los archivos y figuras al Drive
if GEIH_2026_EN_COLAB:
    import glob
    _destino = "/content/drive/MyDrive/GEIH/figuras"
    os.makedirs(_destino, exist_ok=True)
    for _f in glob.glob("geih_2026_*.png"):
        shutil.copy(_f, _destino)
    for _f in (geih_2026_ruta_limpio, geih_2026_ruta_esc):
        shutil.copy(_f, "/content/drive/MyDrive/GEIH/")
    print("Copia en Drive: /content/drive/MyDrive/GEIH/datos_limpios.parquet")

In [ ]:
# GEIH_2026: resumen de cifras clave para redactar el documento breve
geih_2026_resumen = pd.Series({
    "Filas (personas)": f"{len(geih_2026_df_limpio):,}",
    "Columnas finales": f"{geih_2026_df_limpio.shape[1]}",
    "Train / test": f"{int((geih_2026_df_limpio['particion'] == 'train').sum()):,} / {int((geih_2026_df_limpio['particion'] == 'test').sum()):,}",
    "Muestra de modelado (ocupados con ingreso > 0)": f"{int(geih_2026_df_limpio['muestra_modelado'].sum()):,}",
    "Ocupados sin ingreso declarado (excluidos)": f"{geih_2026_n_sin_ingreso:,}",
    "Ocupados con ingreso 0 (excluidos)": f"{geih_2026_n_ingreso_cero:,}",
    "Horas > 112 h convertidas a nulo": f"{geih_2026_n_horas_imposibles:,}",
    "Valores 98/99 de otros ingresos a nulo": f"{geih_2026_n_codigos_otros:,}",
    "Mediana de horas usada para imputar (train)": f"{geih_2026_med_horas:.1f}",
}, name="valor").to_frame()
geih_2026_resumen

## 9. Tabla de decisiones

Esta tabla resume cada decisión de la fase con su justificación. Es la base del documento breve que pide el enunciado.

In [ ]:
pd.set_option("display.max_colwidth", None)
geih_2026_df_decisiones = pd.DataFrame([
    ("Duplicados", "0 repetidas por llave tras la carga; posibles filas idénticas por coincidencia",
     "Conservar", "Una persona es única por DIRECTORIO + SECUENCIA_P + ORDEN; filas iguales en todo son personas distintas, no copias"),
    ("semanas_buscando_trabajo", "Más del 90 % de nulos estructurales; códigos 998/999",
     "998/999 a nulo; el resto de nulos no se imputa", "Solo aplica a quien busca empleo; imputarla inventaría datos y delataría el estado ocupacional"),
    ("ingreso_otras_fuentes", "75 % de vacíos; códigos 98/99 sumados como pesos",
     "98/99 a nulo antes de sumar; vacío a 0 + bandera recibe_otros_ingresos; log1p; Min-Max", "Vacío significa que no recibió otros ingresos; rellenar con media inventaría ingresos; con ~75 % de ceros el IQR es ~0"),
    ("ingreso_laboral (objetivo)", "Nulos estructurales, ocupados sin dato y ceros",
     "No se imputa; muestra_modelado = ocupados con ingreso > 0", "Imputar el objetivo contamina la evaluación; el ingreso 0 es de ocupados sin remuneración"),
    ("horas_trabajadas", "Valores de hasta 130 h; atípicos IQR en más de 1 de cada 7 ocupados",
     "> 112 h a nulo; nulos reales de ocupados con mediana de train; Robust Scaler", "112 h = 16 h x 7 días; la mediana es robusta y la media se contaminaría; Robust no se distorsiona con los atípicos que se conservan"),
    ("ingreso_laboral (atípicos)", "7,6 % sobre el límite del IQR",
     "Conservar + log1p + bandera ingreso_laboral_atipico", "Son personas reales y bien pagadas; el log reduce la asimetría sin perder información"),
    ("edad, cuartos, personas_hogar", "Sin nulos; atípicos IQR que son valores reales",
     "Conservar; mediana de train como respaldo; escalado Z-score (edad) y Robust (cuartos, personas)", "Hogares grandes y personas mayores existen; no son errores"),
    ("sexo, departamento, estado_civil, tipo_vivienda", "Categorías nominales sin orden",
     "Moda de train si hay nulos; One-Hot con drop_first=True", "No hay jerarquía entre categorías; drop_first evita multicolinealidad"),
    ("acceso_servicios, nivel_educativo", "Categorías con orden natural",
     "Ordinal Encoding (Ninguno < Parcial < Completo; Sin educación < ... < Posgrado)", "Conserva la jerarquía; el agrupamiento educativo debe verificarse contra el diccionario"),
    ("Partición", "Evitar data leakage",
     "train_test_split 80/20 estratificado por estado ocupacional, semilla 42, columna particion", "Medianas, modas y escaladores se calculan solo con train; la Fase C reutiliza el mismo split"),
    ("Escalado", "Variables en escalas muy distintas",
     "RobustScaler, StandardScaler y MinMaxScaler según la variable, ajustados con train", "Cada escalador se elige según la distribución; Fase C ajusta el suyo dentro de un Pipeline"),
    ("Archivos de salida", "El enunciado prohíbe normalizar antes del split",
     "datos_limpios.parquet sin escalar + datos_limpios_escalado.parquet", "Permite a la Fase C ajustar el escalado correctamente y deja a la vista el resultado de la normalización"),
    ("Columnas no predictoras", "El objetivo y lo derivado de él podrían colarse como predictoras",
     "ingreso_laboral, log_ingreso_laboral e ingreso_laboral_atipico quedan como objetivo; semanas_buscando_trabajo e ingreso_otras_fuentes quedan como referencia",
     "Evita data leakage con el objetivo; la primera solo existe para desempleados y la segunda se usa en su versión logarítmica"),
], columns=["Columna / paso", "Problema detectado", "Decisión", "Justificación"])
geih_2026_df_decisiones

### Siguiente fase
La **Fase C** (modelos) debe leer `entrega/datos_limpios.parquet`, filtrar `muestra_modelado == True`, usar solo las columnas predictoras (sin las de control, objetivo y referencia), separar train y test con la columna `particion`, y ajustar el escalado dentro de un `Pipeline` solo con train. El objetivo propuesto es `log_ingreso_laboral` (regresión); las predicciones se devuelven a pesos con `np.expm1`.